In [ ]:
from pyspark.sql import functions as F

CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = "bronze_audio_features"

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")


In [ ]:
import os, json

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/audio_features_bulk"
    if not os.path.exists(entity_path): return rows
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                for feat in data.get("audio_features", []):
                    if not feat: continue
                    rows.append({
                        "track_id": feat.get("id"), "danceability": feat.get("danceability"),
                        "energy": feat.get("energy"), "key": feat.get("key"),
                        "loudness": feat.get("loudness"), "mode": feat.get("mode"),
                        "speechiness": feat.get("speechiness"), "acousticness": feat.get("acousticness"),
                        "instrumentalness": feat.get("instrumentalness"), "liveness": feat.get("liveness"),
                        "valence": feat.get("valence"), "tempo": feat.get("tempo"),
                        "duration_ms": feat.get("duration_ms"), "time_signature": feat.get("time_signature"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows).dropDuplicates(["track_id"])
    df = df.withColumn("processing_date", F.current_date())
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
